# MoCA3D on KITTI — Kaggle GPU training

Trains **MoCA3D** (monocular 3D cuboid-corner prediction) on **KITTI**, sized to fit inside Kaggle's free GPU session limits (12h wall clock, ~30 GPU-hours/week).

Repo: https://github.com/jeoncwcw/MoCA3D

## Data sourcing strategy

MoCA3D expects **Omni3D-format** annotation JSON (`KITTI_train.json`, ..., with `bbox3D_cam`, `K`, `dimensions`, `R_cam`, `projected_corners`, ...). The official Omni3D release only distributes these as a pre-built download (gated/manual). Kaggle, however, already hosts **raw KITTI** (images + `label_2` + `calib`) as a public dataset — see `notebooks/kitti-startbook.ipynb` in this repo for reference. This notebook converts that raw KITTI data into Omni3D-format JSON itself, so the only manual download left is the DINOv3 backbone checkpoint.

### ⚠️ Assumptions baked into the converter — review before trusting results

| Choice | What this notebook does | Why / what to change |
|---|---|---|
| Train/val/test split | Deterministic seeded random 80/10/10 split over all 7481 labeled images | **Not** the official Omni3D/KITTI benchmark split. Fine for getting the pipeline running; swap in an official split's image-id list if you need paper-comparable numbers. |
| Category set | Keeps `Car`, `Pedestrian`, `Cyclist` only | KITTI's 3 officially benchmarked classes. `Van`, `Truck`, `Tram`, `Person_sitting`, `Misc`, `DontCare` are dropped. Edit `KEEP_CATEGORIES` to change this. |
| `occluded` → `visibility` | 0→1.0, 1→0.5, 2→0.15, 3→1.0 | KITTI has no continuous visibility score; this is a heuristic so `data_utils.filtered_annotations`'s visibility filter behaves sensibly. Not an official mapping. |
| Camera frame | 3D corners are shifted from KITTI's native camera-0-rectified frame into camera 2's own frame, so a plain 3×3 `K = P2[:, :3]` projects them correctly | Necessary correctness fix — see the conversion cell's comments. Verified against the reprojection sanity-check cell before you train on it. |

If any of these don't match what you need, this is the cell to edit — everything downstream is unchanged from the raw-JSON pipeline.

## Before you run this

1. **Attach a KITTI 3D Object Detection dataset** via **+ Add Input** (e.g. search "KITTI" on Kaggle — the well-known `klemenko/kitti-dataset` has the `training/{image_2,label_2,calib}` layout this notebook expects). The notebook auto-detects the mounted path, so the exact dataset name doesn't matter as long as that layout is present somewhere under `/kaggle/input`.
2. **Attach the DINOv3 ViT-L/16 checkpoint** as a private dataset (gated download from github.com/facebookresearch/dinov3 — Kaggle can't fetch this for you).
3. Set **Settings → Accelerator → GPU T4 x2** (or P100).
4. Run cells top to bottom. Use **Save Version → Save & Run All (Commit)** to keep training running in the background if you close the tab.

Cross-references like `[train.py:293]` point at file/line in the cloned repo.

## 1. Confirm GPU and environment

In [ ]:
import torch, subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("torch:", torch.__version__, "| torch CUDA build:", torch.version.cuda)

assert torch.cuda.device_count() >= 1, "No GPU attached — set Settings > Accelerator > GPU before continuing."


## 2. Clone the repository

In [ ]:
import os, pathlib, shutil, subprocess

WORKING = pathlib.Path("/kaggle/working")
REPO = WORKING / "MoCA3D"
MARKER = REPO / "tools" / "train.py"   # only present in a real, complete clone


def sh(cmd, cwd=None):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0 and result.stderr.strip():
        print(result.stderr)
    return result.returncode == 0


if MARKER.exists():
    # Repo already present from an earlier cell run in this session (or this cell
    # ran twice) - update in place instead of re-cloning.
    print(f"Repo already present at {REPO} and looks complete - pulling latest.")
    pulled = sh(["git", "pull", "--ff-only"], cwd=REPO)
    if not pulled:
        print("git pull failed (see above) - continuing with the existing checkout as-is.")
else:
    if REPO.exists():
        # Directory exists but is missing key files - e.g. a kernel restart left an
        # empty/partial folder behind, or a previous clone was interrupted. This is
        # exactly the "python: can't open file 'data/preprocess/...'" failure mode:
        # cd succeeds because the folder exists, but the repo was never actually cloned into it.
        print(f"{REPO} exists but looks incomplete (missing {MARKER.relative_to(REPO)}) - removing and re-cloning.")
        shutil.rmtree(REPO)
    sh(["git", "clone", "https://github.com/jeoncwcw/MoCA3D.git", str(REPO)], cwd=WORKING)

assert MARKER.exists(), (
    f"Clone did not produce {MARKER} - check the git output above for a network/auth error "
    "before running any later cells."
)

os.chdir(REPO)
print("cwd now:", pathlib.Path.cwd())
sh(["git", "log", "-1", "--oneline"], cwd=REPO)


## 2b. Patch a Python-version compatibility issue in the cloned repo

The repo's own `environment.yml` targets Python 3.11, but Kaggle's kernel may be older. Several vendored/repo files use the `X | None` union type-hint syntax (PEP 604) directly in function signatures, e.g. `data/data_utils.py`:

```python
def get_wds_style_weights(found_datasets, fallback_counts: Dict[str, int] | None = None) -> dict:
```

On Python < 3.10 this raises at **import time** (not a syntax error, a `TypeError` when the annotation is evaluated):

```
TypeError: unsupported operand type(s) for |: '_GenericAlias' and 'NoneType'
```

The fix is to add `from __future__ import annotations` to each affected file, which defers annotations to strings instead of evaluating them (PEP 563) - zero behavior change, safe on every Python version. Two files already have this guard; this cell adds it to the other eight. It's idempotent (safe to re-run, and skips files that already have the fix, e.g. if upstream merges this someday).


In [ ]:
import pathlib, py_compile

# (anchor_text, replacement_text) pairs - anchor is each file's current header,
# replacement is the same header with the future-import inserted before it.
PATCHES = {
    "tools/evaluate.py": (
        "import argparse\nimport os\nimport signal\nimport sys\nfrom pathlib import Path",
        "from __future__ import annotations\n\nimport argparse\nimport os\nimport signal\nimport sys\nfrom pathlib import Path",
    ),
    "models/dinov3/vision_transformer.py": (
        "# the terms of the DINOv3 License Agreement.\n\nimport logging",
        "# the terms of the DINOv3 License Agreement.\n\nfrom __future__ import annotations\n\nimport logging",
    ),
    "models/dinov3/layers/patch_embed.py": (
        "# the terms of the DINOv3 License Agreement.\n\nimport math\nfrom typing import Callable, Tuple, Union",
        "# the terms of the DINOv3 License Agreement.\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Callable, Tuple, Union",
    ),
    "models/dinov3/layers/rope_position_encoding.py": (
        "# the terms of the DINOv3 License Agreement.\n\nimport math\nfrom typing import Literal",
        "# the terms of the DINOv3 License Agreement.\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Literal",
    ),
    "models/dinov3/layers/block.py": (
        "# the terms of the DINOv3 License Agreement.\n\nfrom typing import Callable, List, Optional",
        "# the terms of the DINOv3 License Agreement.\n\nfrom __future__ import annotations\n\nfrom typing import Callable, List, Optional",
    ),
    "models/dinov3/layers/attention.py": (
        "# the terms of the DINOv3 License Agreement.\n\nimport math\nfrom typing import List, Tuple",
        "# the terms of the DINOv3 License Agreement.\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import List, Tuple",
    ),
    "data/preprocess/fill_missing_bbox2d_sam2.py": (
        "import argparse\nimport json\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path",
        "from __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path",
    ),
    "data/data_utils.py": (
        "from pathlib import Path\nimport numpy as np\nfrom typing import Dict, List, Union",
        "from __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\nfrom typing import Dict, List, Union",
    ),
}

patched, already_ok, missing_anchor = [], [], []
for rel_path, (anchor, replacement) in PATCHES.items():
    fp = pathlib.Path(rel_path)
    if not fp.exists():
        missing_anchor.append(rel_path)
        continue
    text = fp.read_text(encoding="utf-8")
    if "from __future__ import annotations" in text:
        already_ok.append(rel_path)
        continue
    if anchor not in text:
        missing_anchor.append(rel_path)
        continue
    fp.write_text(text.replace(anchor, replacement, 1), encoding="utf-8")
    patched.append(rel_path)

print("Patched:", patched)
print("Already had the fix:", already_ok)
if missing_anchor:
    print("WARNING - anchor text not found (repo changed upstream?):", missing_anchor)

# Compile-check every touched file under THIS kernel's actual Python version.
for rel_path in PATCHES:
    py_compile.compile(rel_path, doraise=True)
print("All patched files compile cleanly on this Python version.")


## 3. Install extra Python packages

Kaggle already ships a `torch` + `torchvision` build matched to its driver and CUDA version. **Don't** reinstall torch to the README's pinned `2.9.0+cu128` — that's for a clean conda env, not Kaggle's image.

In [ ]:
!pip install -q omegaconf==2.3.0 safetensors==0.7.0 timm==1.0.24 webdataset==1.0.2 opencv-python==4.11.0.86


## 4. (Optional) PyTorch3D — only if training MoCA3D-Cube

Base MoCA3D training (`tools/train.py`) never imports PyTorch3D. **Skip this cell** unless you're going past the base corner/depth model to the Cube stage — building it from source takes 20–40 minutes.

In [ ]:
RUN_PYTORCH3D_INSTALL = False  # flip to True only if you need MoCA3D-Cube

if RUN_PYTORCH3D_INSTALL:
    !pip install -q fvcore iopath
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@V0.7.8" --no-build-isolation
else:
    print("Skipped — base MoCA3D training does not need PyTorch3D.")


## 5. Locate the attached data

Auto-detects the raw-KITTI directory layout under `/kaggle/input` (works regardless of which specific KITTI dataset you attached) and the DINOv3 checkpoint, then wires the DINOv3 checkpoint into the repo's expected location. Image data is symlinked, not copied, to avoid duplicating ~12GB.

In [ ]:
import glob, os, shutil, pathlib

ROOT = pathlib.Path("/kaggle/working/MoCA3D")
(ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(ROOT / "datasets" / "Omni3D").mkdir(parents=True, exist_ok=True)

# --- locate raw KITTI (image_2 / label_2 / calib) wherever it's mounted ---
label_candidates = glob.glob("/kaggle/input/**/training/label_2", recursive=True)
image_candidates = glob.glob("/kaggle/input/**/training/image_2", recursive=True)
calib_candidates = glob.glob("/kaggle/input/**/training/calib", recursive=True)

assert label_candidates and image_candidates and calib_candidates, (
    "Could not find KITTI training/{label_2,image_2,calib} under /kaggle/input. "
    "Attach a KITTI 3D Object Detection dataset via + Add Input (e.g. 'klemenko/kitti-dataset')."
)

LABEL_SRC = pathlib.Path(label_candidates[0])
IMAGE_SRC = pathlib.Path(image_candidates[0])
CALIB_SRC = pathlib.Path(calib_candidates[0])
print("label_2:", LABEL_SRC)
print("image_2:", IMAGE_SRC)
print("calib  :", CALIB_SRC)

# symlink images into the repo's documented dataset layout (README Section 2)
kitti_dst = ROOT / "datasets" / "KITTI_object" / "training" / "image_2"
kitti_dst.parent.mkdir(parents=True, exist_ok=True)
if not kitti_dst.exists():
    os.symlink(IMAGE_SRC, kitti_dst)

# --- locate the DINOv3 checkpoint you attached separately ---
# Tries the exact expected filename first, then falls back to any .pth under a
# dataset whose name/path mentions dinov3, so a differently-named upload still works.
dinov3_candidates = glob.glob("/kaggle/input/**/dinov3_vitl16_pretrain_lvd1689m*.pth", recursive=True)
if not dinov3_candidates:
    dinov3_candidates = [
        p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
        if "dinov3" in p.lower()
    ]

if not dinov3_candidates:
    print("No DINOv3 checkpoint found under /kaggle/input.")
    print("Here is what IS attached, for reference:")
    for entry in sorted(glob.glob("/kaggle/input/*")):
        print(" -", entry)
    all_pth = (
        glob.glob("/kaggle/input/**/*.pth", recursive=True)
        + glob.glob("/kaggle/input/**/*.safetensors", recursive=True)
    )
    if all_pth:
        print("")
        print(".pth / .safetensors files found (none matched 'dinov3' by name):")
        for f in all_pth:
            print(" -", f)
    error_lines = [
        "DINOv3 checkpoint not found under /kaggle/input.",
        "1) Download 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth' from",
        "   github.com/facebookresearch/dinov3 (requires their access request/approval).",
        "2) Upload it as a private Kaggle Dataset.",
        "3) Attach that dataset here via + Add Input, then re-run this cell.",
        "If it is already attached under a different filename, either rename the file",
        "in your dataset to include 'dinov3', or edit the glob pattern above to match it.",
    ]
    raise AssertionError(chr(10).join(error_lines))

shutil.copy(dinov3_candidates[0], ROOT / "checkpoints" / "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")
print("DINOv3 checkpoint:", dinov3_candidates[0])


## 6. Convert raw KITTI (`label_2` + `calib`) into Omni3D-format JSON

KITTI's label location `(x, y, z)` and rotation are given in **camera 0's rectified frame**, not camera 2's (the `image_2` camera) — but `image_2` is projected with `P2`, and MoCA3D's own `dataprocess.py` projects corners with a plain 3×3 `K @ [X, Y, Z]`, which only gives the right pixel if `[X, Y, Z]` is already in **camera 2's own frame**.

`P2 = [K | K·t2]`, so `t2 = K⁻¹ · P2[:, 3]`. This cell shifts every corner by `t2` before saving, so the `K = P2[:, :3]` written into the JSON is directly usable by the rest of this repo (Steps 1–2 in the architecture walkthrough), and matches what a well-established, independent KITTI toolkit (`kuixu/kitti_object_vis`) does. The corner template itself (`compute_box_3d`, y ranges `0` to `-h`) is the standard KITTI convention and was cross-checked against that same reference.

In [ ]:
import json
import numpy as np
from PIL import Image


def roty(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def read_calib_P2(calib_path):
    with open(calib_path) as f:
        for line in f:
            if line.startswith("P2:"):
                vals = [float(x) for x in line.split(":", 1)[1].split()]
                return np.array(vals, dtype=np.float64).reshape(3, 4)
    raise ValueError(f"No P2 line found in {calib_path}")


def read_label_lines(label_path):
    objs = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split(" ")
            if len(parts) < 15:
                continue
            objs.append({
                "type": parts[0],
                "truncated": float(parts[1]),
                "occluded": int(float(parts[2])),
                "bbox": [float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])],
                "h": float(parts[8]), "w": float(parts[9]), "l": float(parts[10]),
                "loc": [float(parts[11]), float(parts[12]), float(parts[13])],
                "ry": float(parts[14]),
            })
    return objs


def kitti_box_to_cam2_corners(obj, t2):
    """8 corners of the 3D box in camera-2's own frame (matches K = P2[:, :3] directly)."""
    h, w, l = obj["h"], obj["w"], obj["l"]
    x_c = [l / 2, l / 2, -l / 2, -l / 2, l / 2, l / 2, -l / 2, -l / 2]
    y_c = [0, 0, 0, 0, -h, -h, -h, -h]          # KITTI: Y down, loc = bottom-face center
    z_c = [w / 2, -w / 2, -w / 2, w / 2, w / 2, -w / 2, -w / 2, w / 2]
    R = roty(obj["ry"])
    corners_rect0 = R @ np.array([x_c, y_c, z_c]) + np.array(obj["loc"]).reshape(3, 1)
    corners_cam2 = corners_rect0 + np.array(t2).reshape(3, 1)
    return corners_cam2.T  # (8, 3)


KEEP_CATEGORIES = {"Car": 0, "Pedestrian": 1, "Cyclist": 2}   # edit to widen/narrow the class set
OCCLUDED_TO_VISIBILITY = {0: 1.0, 1: 0.5, 2: 0.15, 3: 1.0}     # heuristic — see markdown above
SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1                                 # remainder -> test

label_ids = sorted(p.stem for p in LABEL_SRC.glob("*.txt"))
rng = np.random.default_rng(SEED)
shuffled = list(label_ids)
rng.shuffle(shuffled)
n = len(shuffled)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)
split_ids = {
    "train": shuffled[:n_train],
    "val": shuffled[n_train:n_train + n_val],
    "test": shuffled[n_train + n_val:],
}
print("total labeled images:", n, "| split sizes:", {k: len(v) for k, v in split_ids.items()})

categories = [{"id": cid, "name": name, "supercategory": name} for name, cid in KEEP_CATEGORIES.items()]
OMNI3D_OUT = ROOT / "datasets" / "Omni3D"

for split, ids in split_ids.items():
    images, annotations = [], []
    ann_id = 0
    for img_id_str in ids:
        img_id = int(img_id_str)
        calib_path = CALIB_SRC / f"{img_id_str}.txt"
        label_path = LABEL_SRC / f"{img_id_str}.txt"
        image_path = IMAGE_SRC / f"{img_id_str}.png"
        if not (calib_path.exists() and label_path.exists() and image_path.exists()):
            continue

        P2 = read_calib_P2(calib_path)
        K = P2[:, :3]
        t2 = np.array([P2[0, 3] / K[0, 0], P2[1, 3] / K[1, 1], 0.0])

        with Image.open(image_path) as im:
            width, height = im.size

        images.append({
            "id": img_id,
            "dataset_id": 0,
            "width": width,
            "height": height,
            "file_path": f"KITTI_object/training/image_2/{img_id_str}.png",
            "K": K.tolist(),
            "src_90_rotate": 0,
            "src_flagged": False,
        })

        for obj in read_label_lines(label_path):
            if obj["type"] not in KEEP_CATEGORIES:
                continue
            if obj["h"] <= 0 or obj["w"] <= 0 or obj["l"] <= 0:
                continue

            corners_cam2 = kitti_box_to_cam2_corners(obj, t2)   # (8, 3) meters, camera-2 frame
            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": KEEP_CATEGORIES[obj["type"]],
                "category_name": obj["type"],
                "valid3D": True,
                "bbox2D_tight": obj["bbox"],
                "bbox3D_cam": corners_cam2.tolist(),
                "center_cam": corners_cam2.mean(axis=0).tolist(),
                "dimensions": [obj["w"], obj["h"], obj["l"]],       # Omni3D order: width, height, length
                "R_cam": roty(obj["ry"]).tolist(),
                "behind_camera": bool(np.any(corners_cam2[:, 2] <= 0)),
                "truncation": obj["truncated"],
                "visibility": OCCLUDED_TO_VISIBILITY.get(obj["occluded"], 1.0),
            })
            ann_id += 1

    out = {
        "info": {"id": f"KITTI_{split}", "source": 0, "name": "KITTI", "split": split, "version": "1.0", "url": ""},
        "images": images,
        "categories": categories,
        "annotations": annotations,
    }
    out_path = OMNI3D_OUT / f"KITTI_{split}.json"
    with open(out_path, "w") as f:
        json.dump(out, f)
    print(f"{split:5s}: {len(images):5d} images, {len(annotations):6d} annotations -> {out_path}")


## 7. Visual sanity check — did the conversion come out right?

Reprojects the converted 3D corners (red wireframe) onto a sample image using the `K` this notebook just wrote, alongside KITTI's own tight 2D box (green). The red wireframe should sit tightly around each car — if it's offset or the wrong size, something in Cell 6 needs fixing before you spend GPU hours training on it.

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample = json.load(open(OMNI3D_OUT / "KITTI_train.json"))
img_entry = next(im for im in sample["images"]
                  if any(a["image_id"] == im["id"] for a in sample["annotations"]))
anns = [a for a in sample["annotations"] if a["image_id"] == img_entry["id"]]

img = cv2.cvtColor(cv2.imread(str(ROOT / "datasets" / img_entry["file_path"])), cv2.COLOR_BGR2RGB)
K = np.array(img_entry["K"])
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]

vis = img.copy()
for a in anns:
    corners = np.array(a["bbox3D_cam"])
    proj = (K @ corners.T).T
    proj = proj[:, :2] / proj[:, 2:3]
    for i, j in EDGES:
        cv2.line(vis, tuple(proj[i].astype(int)), tuple(proj[j].astype(int)), (255, 0, 0), 2)
    x1, y1, x2, y2 = map(int, a["bbox2D_tight"])
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 1)

plt.figure(figsize=(14, 5))
plt.imshow(vis)
plt.title(f"image {img_entry['id']} | red = reprojected 3D box (this notebook) | green = KITTI bbox2D_tight")
plt.axis("off")
plt.show()


## 7b. (Optional / Option B) Use official pre-converted Omni3D KITTI JSON instead

If you already have the official `KITTI_train.json` / `KITTI_val.json` / `KITTI_test.json` from Omni3D's own release (gated/manual download), attach them as a Kaggle dataset and copy them in here — this **overwrites** the self-converted files from Cell 6 with the official ones, giving you paper-comparable splits and categories. Leave `RUN_OPTION_B = False` to keep using this notebook's own conversion.

In [ ]:
RUN_OPTION_B = False

if RUN_OPTION_B:
    OMNI3D_KITTI_DIR = "/kaggle/input/omni3d-kitti-annotations"  # edit to your attached dataset's path
    for split in ["train", "val", "test"]:
        shutil.copy(f"{OMNI3D_KITTI_DIR}/KITTI_{split}.json", OMNI3D_OUT / f"KITTI_{split}.json")
    print("Replaced with official Omni3D KITTI annotations.")
else:
    print("Skipped — using this notebook's own conversion from Cell 6.")


## 8. Run the repo's own Step 1–2 preprocessing

Same pipeline as the architecture walkthrough's Steps 1–2: project corners (a no-op here since Cell 6 already wrote `bbox3D_cam`, but this also derives `projected_corners`/`depth`), assign quality groups, canonicalize corner ordering.

In [ ]:
%cd /kaggle/working/MoCA3D

import pathlib

expected = [pathlib.Path("datasets/Omni3D") / f"KITTI_{s}.json" for s in ("train", "val", "test")]
missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing " + ", ".join(missing) + ". "
        "Run Section 5 (locate attached data) and Section 6 (convert raw KITTI to Omni3D JSON) "
        "in this session before this cell - a fresh clone/session does not carry datasets/ over "
        "from a previous run, so they need to be (re-)run every time this notebook starts fresh."
    )
print("Found all expected Omni3D JSON files - proceeding with preprocessing.")

!python data/preprocess/dataprocess.py --input_dir datasets/Omni3D --output_dir datasets/MoCA3D
!python data/preprocess/build_quality_groups.py --root_dir datasets/MoCA3D
!python data/preprocess/ordering_new.py --root_dir datasets/MoCA3D
!ls -la datasets/MoCA3D


## 9. Sanity check before spending GPU time

In [ ]:
from collections import Counter

d = json.load(open("datasets/MoCA3D/KITTI_train.json"))
print("annotations:", len(d["annotations"]))
print("quality distribution:", Counter(o.get("quality") for o in d["annotations"]))

img_info = d["images"][0]
Image.open(f"datasets/{img_info['file_path']}").verify()
print("image path OK:", img_info["file_path"])


## 10. Build a config sized for a Kaggle session

The default config (`num_epochs: 120`, `epoch_length: 50000`) is a multi-GPU-day job. Trim it down; raise these once you've measured real throughput in Cell 11.

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load("configs/MoCA_config.yaml")
cfg.num_epochs = 8            # short run to fit one session; raise later
cfg.warmup_epochs = 1
cfg.batch_size = 8            # T4/P100 = 16GB; raise if it fits, lower on OOM
cfg.grad_accum_steps = 2
cfg.num_workers = 2
cfg.save_interval = 1
cfg.val_interval = 1
cfg.model_name = "MoCA3D_kitti_kaggle"

OmegaConf.save(cfg, "configs/MoCA_kaggle.yaml")
print(OmegaConf.to_yaml(cfg))


## 11. Train

- `--no-compile` skips `torch.compile` warm-up, not worth it for a short run.
- `--train-epoch-length` overrides `epoch_length` directly ([train.py:205](../tools/train.py)).
- For an unattended run past your browser tab closing, use **Save Version → Save & Run All (Commit)** instead of running this cell interactively.

In [ ]:
%cd /kaggle/working/MoCA3D
!python tools/train.py \
    --config configs/MoCA_kaggle.yaml \
    --train-loader image --val-loader image \
    --train-datasets KITTI --val-datasets KITTI \
    --train-epoch-length 3000 \
    --val-epoch-length 500 \
    --no-compile \
    --allow-existing-checkpoint-dir


## 12. Persist checkpoints for the next session

Kaggle wipes the VM every session — only `/kaggle/working/` survives as the notebook's committed **Output**. Commit the notebook so these checkpoints are saved; attach that output as an input dataset next time.

In [ ]:
!ls -la checkpoints/MoCA3D_kitti_kaggle/
!du -sh checkpoints/MoCA3D_kitti_kaggle/


## 13. Resume in a later session

**Caveat:** `--init-checkpoint` only loads model *weights* ([train.py:293-309](../tools/train.py)). The optimizer and LR scheduler restart fresh every session — a warm restart, not a true resume.

In [ ]:
PREV_CHECKPOINT = "/kaggle/input/<your-previous-notebook-output>/checkpoints/MoCA3D_kitti_kaggle/best_uv.pth"

dst_dir = pathlib.Path("checkpoints/MoCA3D_kitti_kaggle")
dst_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(PREV_CHECKPOINT, dst_dir / "warm_start.pth")
print("Copied to", dst_dir / "warm_start.pth")


In [ ]:
!python tools/train.py \
    --config configs/MoCA_kaggle.yaml \
    --train-loader image --val-loader image \
    --train-datasets KITTI --val-datasets KITTI \
    --train-epoch-length 3000 --val-epoch-length 500 \
    --no-compile --allow-existing-checkpoint-dir \
    --init-checkpoint checkpoints/MoCA3D_kitti_kaggle/warm_start.pth


## 14. Evaluate

`evaluate.py`'s `--split` flag only accepts `"test"` and defaults to it ([evaluate.py:411](../tools/evaluate.py)) — it always reads `KITTI_test.json`.

In [ ]:
!python tools/evaluate.py \
    --config configs/MoCA_kaggle.yaml \
    --loader image \
    --datasets KITTI \
    --checkpoint checkpoints/MoCA3D_kitti_kaggle/best_uv.pth


## 15. (Bonus) Visualize one prediction

Loads the trained checkpoint directly, runs it on a validation image, and plots predicted vs. ground-truth corner heatmaps with `utils.functions.visualize_heatmaps` — the same function the training loop's own validation step calls ([engine.py:157-172](../utils/engine.py)).

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/MoCA3D")

from data.image_dataloader import build_image_dataloader
from models.moca_3d import Moca3DModel
from utils.functions import visualize_heatmaps

device = torch.device("cuda:0")
cfg = OmegaConf.load("configs/MoCA_kaggle.yaml")
cfg.device = str(device)
cfg.feature_mode = False   # run the DINOv3 backbone live on raw images

val_loader, _ = build_image_dataloader(
    root_dir=pathlib.Path("datasets/MoCA3D"),
    data_dir=pathlib.Path("datasets/"),
    seed=42,
    split="val",
    batch_size=1,
    dino_image_size=int(cfg.data.dino_image_size),
    target_quality=str(cfg.data.target_quality),
    min_area=int(cfg.data.min_area_object),
    shuffle=True,
    num_workers=0,
    datasets=["KITTI"],
)

model = Moca3DModel(cfg).to(device)
state = torch.load("checkpoints/MoCA3D_kitti_kaggle/best_uv.pth", map_location=device)
model.load_state_dict(state, strict=True)
model.eval()

batch = next(iter(val_loader))
batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

with torch.no_grad():
    out = model(
        bbx2d_tight=batch_gpu["2d_bbx"],
        mask=batch_gpu["padding_mask"],
        images_dino=batch_gpu["image_dino"],
    )

pred_coords_128 = out["corner coords"][0].cpu() / 4.0     # 512-px scale -> 128 heatmap scale
gt_coords_128 = batch_gpu["gt_corners"][0].cpu() * 128.0
heatmaps = out["corner heatmaps"][0].cpu()

visualize_heatmaps(heatmaps, pred_coords_128, gt_coords_128, "prediction_preview.png")

from IPython.display import Image as IPImage, display
display(IPImage("prediction_preview.png"))
